# 25 - FrozenLake DQN with Reasoning

Choose where to apply the value loss independently of the reasoning recurrence.

- `REASONING_LOSS = "all_tokens"` trains a Bellman loss at every generated
  reasoning state. A shared head predicts environment actions plus `think`.
  Environment actions are unavailable before `MIN_REASONING_STEPS`; think is
  unavailable at `MAX_REASONING_STEPS`.
- `REASONING_LOSS = "last_token"` trains only the final state's recorded
  environment action. Set the minimum equal to the maximum for a fixed budget.
  The head has only environment actions, with no learned stopping decision.

`RESIDUAL_CONNECTIONS` adds the input latent to the raw backbone output.
`WRAP_LATENT` selects `"none"`, `"step"`, or `"sequence"`. Step wrapping feeds back
within a decision; sequence wrapping also carries the final latent into the next
environment decision, resetting only at a new sequence. The latent passes through
input RMSNorm followed by the shared Qwen3-style gated input MLP. The same learned
embedding is added AFTER this block at every reasoning token: `input = z_start + MLP(feedback)`.
Feedback is zero at a wrapping-scope boundary, or at every token without wrapping.
The bias-free MLP maps zero to zero, so a reset token receives exactly `z_start`.
With residual addition, the output is `feedback + update`; both flags enabled
accumulate updates from zero. Generated feedback is detached during replay, while
each token's injected embedding and the live additive path remain differentiable.
Zero-feedback-only batches have zero data gradients for the input RMSNorm/MLP.

The first step is free. All-token supervision adds a signed reward for each `think`
transition; last-token supervision adds the total reward `reasoning_reward * (K - 1)` once.
Reported environment rewards exclude these internal rewards. The backbone has no
final output normalization. Generation always uses continuous paged KV caching.

The trainable input RMSNorm uses backbone.rms_norm_eps (Qwen3: 1e-6), followed by a shared Qwen3-style gated input MLP with standard PyTorch initialization. Both operate in FP32. There is no output-side update RMSNorm or projection.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Literal, cast

import torch
from huggingface_hub import HfApi
from tensordict import TensorDict
from torch import nn

from mouse_core import MouseOptimizer
from mouse_core.data import (
    Augmenter,
    DataLoader,
    Tokenizer,
    compose,
    load_stores_from_hub,
    save_tokenizer,
)
from mouse_core.models import (
    LoRAConfig,
    Backbone,
    BaseBackbone,
    Embedding,
    Model,
    MultiEmbedding,
    ReasoningBatch,
    SequentialReasoner,
    save_model,
)
from mouse_core.models.backbone.continuous_decode import ContinuousDecodeSession
from mouse_core.objectives import replay_action_indices, DqnObjective, boundary_discount, final_reasoning_loss, value_reasoning_loss

# Repositories
DATASET_ID = "mouse-example-frozenlake-dataset"
MODEL_ID = "mouse-example-reasoning-dqn"

# Environment
MAX_ACTIONS = 4
MAX_OBS_DISCRETE = 64
MAX_STEPS_PER_EPISODE = 24
REPEAT_COUNT = 20

# Model
PRETRAINED = "Qwen/Qwen3-0.6B"
BACKBONE_LORA = {"rank": 16, "alpha": 32.0, "dropout": 0.0, "dtype": "float32"}

# Tokenizer
COMPUTE_TOKENS = 8

# Reasoning
REASONING_LOSS: Literal["all_tokens", "last_token"] = "all_tokens"
RESIDUAL_CONNECTIONS = True
WRAP_LATENT = "step"
INPUT_MLP_OUTPUT_SCALE = 711.24
MIN_REASONING_STEPS = 1
MAX_REASONING_STEPS = 16
REASONING_REWARD = -0.01
MAX_ACTIVE_REQUESTS = 8

# Training
PREFETCH = 4
SAMPLE_BUDGET = 8
NUM_WORKERS = 0
NUM_CYCLES = 1
TRAIN_STEPS = 10

# Objectives
GAMMA = 0.99
GAMMA_THINK = 1.0

# Optimizer
LR_EMBEDDING = 1e-5
LR_REASONING_INPUT = 1e-5
LR_BACKBONE_LORA = 1e-5
LR_HEADS = 1e-5
MAX_GRAD_NORM = 1.0

# Target updates
POLYAK_TAU_EMBEDDING = 0.1
POLYAK_TAU_REASONING_INPUT = 0.1
POLYAK_TAU_BACKBONE_LORA = 0.1
POLYAK_TAU_VALUE = 0.001

# Random seeds
SEED = 7

# Runtime options
USE_KERNELS = False
GRADIENT_CHECKPOINTING = True

## Runtime


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook trains with FlexAttention and requires CUDA; CPU supports inference only."
    )
device = torch.device("cuda")
torch.manual_seed(SEED)

## Load Data

Use the same recorded dataset and coordinator/agent/environment event schema as
example 02. The dataset contains complete outer tasks with repeated procedural
FrozenLake episodes. Set `MAX_STEPS_PER_EPISODE` and `REPEAT_COUNT` to match the
collector. This notebook never creates or steps an environment.

Only complete outer tasks are sampled. Inner endings and reset observations stay
in the context; `_terminated` and `_truncated` end the outer return. Terminal and
passive reset transitions have no prediction unless they request an action.
Reasoning adds latent computation at decision boundaries without changing the
recorded actions or environment rewards. Evaluate saved checkpoints separately in
[15 — Inference](15_inference.ipynb).


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train")

## Tokenizer

Define the prompt and field formats, then the tokenizer. `input_fields` defines
transformer inputs in sequence order; `objective_fields` keeps training labels
separate. The action-request newline selects the prediction readout.

`COMPUTE_TOKENS` sets a constant number of compute tokens before every value readout.
Set it to zero to disable compute text. The field filters on `action_requested`
and uses the literal format `" ." * COMPUTE_TOKENS`. No callback or runtime context
is needed. The configured text is saved with the tokenizer and reused at inference.

Qwen encodes each spaced period as one token (ID 659); ten bare periods can merge
into just one token. Passive and
terminal events emit no compute text. Compute tokens increase token length without
changing the number of sampled outer episodes.

Only the newline has `head_output="all"`. The value loss trains the compute
positions through causal attention. Notebook 15 uses a fixed inference count so
rebuilding a cached history reproduces its text.

Reasoning uses the same fields and objective labels as example 02. The only schema
difference is `` on the action-request newline. The reasoner
replaces that position with its learned input and inserts additional latent
positions after tokenization; context and recorded actions stay unchanged.


In [ ]:
def action_name(action):
    return ("a", "b", "c", "d")[action]


episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\nx marks no action at the beginning of an episode.\n"
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED, "reasoning": {"type": "vocabulary", "tokens": ["reasoning"]}},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "episode {repeat_episode_index+1}\n",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "x",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "format": "{action_name(action)}",
            "format_functions": {"action_name": action_name},
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ",{observation}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ",r={reward:g}",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": " ." * COMPUTE_TOKENS,
        },
        {
            "type": "token",
            "tokenizer": "reasoning",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "reasoning",
            "head_output": "last",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": ["inner_terminated", "inner_truncated"],
            "format": "\n\n",
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
    ],
)
# Validate the existing single-token decision marker before allocating the model.
tokenizer({"provider": "environment", "payload": {"action_requested": True}})


## Replay

Sample complete outer tasks, retaining their repeated episodes. Each sample receives
its own sequence ID so attention and objective transitions cannot cross tasks.

Apply augmentation to event payloads before tokenization. Action permutations also
update `_q_star`, keeping labels aligned. Pass the composed transform to the loader;
`next_batch()` returns `(inputs, objective_data)`, including prediction-to-step and
sequence mappings. Inference uses the saved tokenizer without augmentation.

`SAMPLE_BUDGET` counts complete outer episodes, including all inner repeats.
Each batch contains eight samples regardless of their token lengths. Compute tokens
increase token length without reducing the number of sampled episodes.

After sampling and augmentation, reasoning generation inserts latent positions
at the marked decisions. It preserves the recorded environment transitions and
actions. It does not interact with the environment.


In [ ]:
augmenter = Augmenter(
    seed_field="episode_index",
    fields=[
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "action",
            "input_vector_field": "_q_star",
            "vocab_size": MAX_ACTIONS,
        },
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
        },
    ],
)

train_transform = compose(stages=(augmenter.events, tokenizer.encode))


def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Train only on episodes with a recorded final environment result."""
    first, last = events[0], events[-1]
    return (first["provider"] == "coordinator" and first["step_index"] == 0
            and first["payload"].get("_episode_start") is True
            and last["provider"] == "environment") and (
        last["payload"].get("_terminated") is True
        or last["payload"].get("_truncated") is True
    )


loader = DataLoader(
    stores=stores,
    sample_budget=SAMPLE_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    transform=train_transform,
    prefetch=PREFETCH,
    num_workers=NUM_WORKERS,
    seed=SEED,
)

## Build Encoder and Heads

`REASONING_LOSS` selects an A+1 action head with adaptive depth or an A-action
head with a fixed budget. `RESIDUAL_CONNECTIONS` and `WRAP_LATENT` independently control the
residual addition and feedback. `SequentialReasoner` owns the trainable input RMSNorm
and a shared Qwen3-style gated input MLP with standard PyTorch initialization.
Online and delayed models replay the same generated history. Only online replay
receives gradients; eval mode disables dropout through backward.


In [ ]:
class ReasoningDqn(Model):
    def __init__(
        self,
        *,
        embedding: MultiEmbedding,
        backbone: BaseBackbone,
        value_head: nn.Module,
        input_mlp_output_scale: float,
        min_steps: int,
        max_steps: int,
        residual_connections: bool,
        wrap_latent: Literal["none", "step", "sequence"],
        reasoning_loss: Literal["all_tokens", "last_token"],
    ) -> None:
        super().__init__()
        if reasoning_loss not in {"all_tokens", "last_token"}:
            raise ValueError("reasoning_loss must be all_tokens or last_token")
        if reasoning_loss == "last_token" and min_steps != max_steps:
            raise ValueError("last_token requires equal minimum and maximum reasoning steps")
        self.reasoning_loss = reasoning_loss
        # Both context and starting token embeddings train during forward.
        self.embedding = embedding
        self.backbone = backbone
        self.value_head = value_head
        self.reasoner = SequentialReasoner(
            hidden_dim=backbone.hidden_dim, input_mlp_intermediate_dim=backbone.mlp_intermediate_dim,
            input_mlp_output_scale=input_mlp_output_scale, min_steps=min_steps, max_steps=max_steps,
            residual_connections=residual_connections, wrap_latent=wrap_latent, adaptive_depth=reasoning_loss == "all_tokens",
         rms_norm_eps=backbone.rms_norm_eps)

    @torch.no_grad()
    def generate(
        self,
        tokens: TensorDict,
        *,
        max_active_requests: int = 8,
        cache: ContinuousDecodeSession | None = None,
    ) -> ReasoningBatch:
        return self.reasoner.generate(
            tokens,
            embedding=self.embedding,
            backbone=self.backbone,
            action_values=self.value_head,
            backend="continuous",
            max_active_requests=max_active_requests,
            cache=cache,
        )

    def forward(self, generated: ReasoningBatch) -> dict[str, torch.Tensor]:
        states = self.reasoner(
            generated,
            embedding=self.embedding,
            backbone=self.backbone,
        )
        rows = torch.arange(len(states), device=states.device)
        if self.reasoning_loss == "last_token":
            return dict(q=self.value_head(states[rows, generated.lengths - 1]), states=states)
        q_steps = self.value_head(states)
        return dict(
            q=q_steps[rows, generated.lengths - 1, :-1],
            q_steps=q_steps,
            states=states,
        )


text_embedding = Embedding(pretrained=PRETRAINED)
reasoning_embedding = Embedding(vocab_size=1, hidden_dim=text_embedding.hidden_dim)
nn.init.normal_(reasoning_embedding.embed_tokens.weight, std=0.02)
embedding = MultiEmbedding(embeddings={"text": text_embedding, "reasoning": reasoning_embedding})
backbone = Backbone(lora=LoRAConfig(**BACKBONE_LORA), residual_dtype=torch.float32, pretrained=PRETRAINED, dtype=torch.float32)
backbone.gradient_checkpointing = GRADIENT_CHECKPOINTING
value_head = nn.Linear(backbone.hidden_dim, MAX_ACTIONS + int(REASONING_LOSS == "all_tokens"))
online = ReasoningDqn(
    embedding=embedding,
    backbone=backbone,
    value_head=value_head,
    input_mlp_output_scale=INPUT_MLP_OUTPUT_SCALE,
    min_steps=MIN_REASONING_STEPS,
    max_steps=MAX_REASONING_STEPS,
    residual_connections=RESIDUAL_CONNECTIONS,
    wrap_latent=WRAP_LATENT,
    reasoning_loss=REASONING_LOSS,
).to(device)
target = online.copy().requires_grad_(False).eval()
if USE_KERNELS:
    backbone.enable_kernels()
    cast(Backbone, target.backbone).enable_kernels()

## Train

Generate the online history, then replay once with online gradients and once
with delayed weights. The delayed model never generates a separate trajectory.

For `all_tokens`, every visited state has one Bellman transition:

`think target = reasoning_reward + gamma_think * max Q_target(next reasoning state, available actions)`

`environment target = reward + gamma * max Q_target(next observation, first reasoning state, available actions)`

Train think at every continuation and the recorded environment action at the
stopping depth. Mask environment actions before the minimum and think at the cap
in generation and targets. No losses train padding or ungenerated depths.

For `last_token`, only the final state's recorded environment action has a loss:

`target = reward + reasoning_reward * (K - 1) + gamma * max Q_target(next observation, final reasoning state)`

The next observation's final state is used in this mode because it is the only
state with a trained value head. The reasoning reward belongs to the current observation's
reasoning and is added once. Both modes handle terminal discounts independently
and leave the recorded environment rewards unchanged.


In [ ]:
optimizer = MouseOptimizer(
    optimizer={"name": "adamw", "betas": (0.9, 0.95), "eps": 1e-8, "weight_decay": 0.0},
    groups={
        "embedding": {
            "online": online.embedding.embeddings["text"],
            "compile": False,
            "lr": LR_EMBEDDING,
            "target": target.embedding.embeddings["text"],
            "polyak_tau": POLYAK_TAU_EMBEDDING,
        },
        "backbone_lora": {
            "online": online.backbone,
            "compile": False,
            "parameters": "lora",
            "lr": LR_BACKBONE_LORA,
            "target": target.backbone,
            "polyak_tau": POLYAK_TAU_BACKBONE_LORA,
        },
        "value_head": {
            "online": online.value_head,
            "compile": False,
            "lr": LR_HEADS,
            "target": target.value_head,
            "polyak_tau": POLYAK_TAU_VALUE,
        },
        "trainable_embeddings": {
            "online": online.embedding.embeddings["reasoning"],
            "compile": False,
            "lr": LR_HEADS,
            "target": target.embedding.embeddings["reasoning"],
            "polyak_tau": POLYAK_TAU_VALUE,
        },
        "reasoning_input": {
            "online": nn.ModuleDict({"norm": online.reasoner.norm, "mlp": online.reasoner.input_projection}),
            "compile": False,
            "lr": LR_REASONING_INPUT,
            "target": nn.ModuleDict({"norm": target.reasoner.norm, "mlp": target.reasoner.input_projection}),
            "polyak_tau": POLYAK_TAU_REASONING_INPUT,
        },
    },
)

objective = DqnObjective(
    reward=None,
    value=None,
    gate=None,
    temperature=0.0,
    discount=boundary_discount(
        gamma_step=GAMMA,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
)


def run_train(
    *, training_step: int, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor], int]:
    """Sample recorded episodes, generate reasoning, and train one batch per step."""
    if type(training_step) is not int or training_step < 0:
        raise ValueError("training_step must be a nonnegative integer")
    if num_steps < 1:
        raise ValueError("num_steps must be positive")
    if not 0 < MAX_GRAD_NORM < float("inf"):
        raise ValueError("Invalid max_grad_norm")
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        tokens, objective_data = loader.next_batch()
        tokens = tokens.to(device)
        objective_data = objective_data.to(device)
        sequence_ids = objective_data["sequence_ids"]
        # Eval mode disables stochastic layers; gradients and optimizer updates remain enabled.
        # Keep it through backward so checkpoint recomputation uses the same mode.
        online.eval()
        target.eval()
        if online.reasoner.max_steps == 1 and online.reasoner.wrap_latent != "sequence":
            from mouse_core.data import start_only_reasoning_episodes, pack_reasoning_episodes
            generated, objective_data = pack_reasoning_episodes(start_only_reasoning_episodes(
                tokens, objective_data, hidden_dim=online.reasoner.hidden_dim,
                max_steps=1, generated_step=training_step))
            generated, objective_data = generated.to(device), objective_data.to(device)
        else:
            generated = online.generate(
                tokens,
                max_active_requests=MAX_ACTIVE_REQUESTS,
            )
            objective_data = generated.align_objective_data(tokens, objective_data)
        # Both forward calls consume the same complete generated token sequences.
        lengths = generated.lengths
        out = online(generated)
        with torch.no_grad():
            delayed = target(generated)
            if online.reasoning_loss == "all_tokens":
                delayed_steps = delayed["q_steps"]
                delayed_q = delayed_steps[:, 0].clone()
                if online.reasoner.min_steps > 1:
                    delayed_q[:, :-1] = -torch.inf
                if online.reasoner.max_steps == 1:
                    delayed_q = delayed_q[:, :-1]
            else:
                delayed_q = delayed["q"]
        states, q = out["states"], out["q"]
        step_ids = objective_data["step_ids"]
        rows, recorded_actions = replay_action_indices(
            objective_data, sequence_ids, step_ids=step_ids, readout_shape=q.shape[:1],
        )
        actions = torch.zeros(len(q), dtype=torch.long, device=q.device)
        actions[rows] = recorded_actions
        selected = q.gather(-1, actions[:, None]).squeeze(-1)
        _, objective_metrics = objective(
            objective_data=objective_data,
            sequence_ids=sequence_ids,
            step_ids=objective_data["step_ids"],
            predictions=selected,
            backup_value_predictions=delayed_q,
            backup_action_predictions=delayed_q,
        )
        weights, backup = (
            objective_metrics["backup_weight"],
            objective_metrics["backup"],
        )
        if not isinstance(weights, torch.Tensor) or not isinstance(
            backup, torch.Tensor
        ):
            raise TypeError("DQN backup values and weights must be tensors")
        if online.reasoning_loss == "all_tokens":
            loss, _ = value_reasoning_loss(
                min_steps=online.reasoner.min_steps,
                q_values=out["q_steps"], target_q_values=delayed_steps, lengths=lengths,
                environment_actions=actions, environment_targets=backup, weights=weights,
                reasoning_reward=REASONING_REWARD, gamma_think=GAMMA_THINK,
            )
        else:
            loss, _ = final_reasoning_loss(
                q_values=selected, environment_targets=backup, lengths=lengths,
                weights=weights, reasoning_reward=REASONING_REWARD,
            )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        norm = nn.utils.clip_grad_norm_(
            online.parameters(), MAX_GRAD_NORM, error_if_nonfinite=True
        )
        optimizer.step(training_step)
        training_step += 1
        metrics = {
            "loss": float(loss.detach()),
            "dqn_loss": float(loss.detach()),
            "reasoning_reward": float(REASONING_REWARD * ((lengths - 1) * weights).sum()
                                    / weights.sum().clamp_min(1)),
            "grad_norm": float(norm),
            "reasoning_state_rms": float(
                states.detach()[torch.arange(len(states), device=device), lengths - 1]
                .square()
                .mean()
                .sqrt()
            ),
            "reasoning_steps": float(lengths.float().mean()),
        }
    assert loss is not None
    return (loss.detach(), metrics, training_step)

## Run

Train on fixed replay, regenerating reasoning for each optimizer update.
The selected loss determines whether every generated state or only the final
state receives value supervision. The residual connection remains independent.


In [ ]:
training_step = 0
try:
    for cycle in range(NUM_CYCLES):
        loss, metrics, training_step = run_train(training_step=training_step, num_steps=TRAIN_STEPS)
        print(f"cycle={cycle} train  loss={loss.item():.4f}  {metrics}")
finally:
    loader.close()

## Save to Hugging Face

Run this cell to upload the online weights, tokenizer files and inference settings
together to `MODEL_ID` in your authenticated Hugging Face account. Temporary files
are only used to assemble the upload. Optimizer and delayed-network state are not
included. Evaluate the uploaded repository in the matching reasoning section of
[15 — Inference](15_inference.ipynb).


In [ ]:
with TemporaryDirectory() as directory:
    output = Path(directory)
    save_model(model=online, path=output)
    save_tokenizer(tokenizer=tokenizer, path=output / "tokenizer")
    (output / "reasoning_settings.json").write_text(
        json.dumps(
            {
                "pretrained": PRETRAINED,
                "backbone_lora": BACKBONE_LORA,
                "environment": {
                    "id": "Procedural-FrozenLake-v1",
                    "width": 8,
                    "height": 8,
                    "max_episode_steps": MAX_STEPS_PER_EPISODE,
                    "slippery_success_rate": 1.0,
                    "permute_obs": True,
                    "permute_actions": True,
                    "emit_q_star": False,
                },
                "repeat_count": REPEAT_COUNT,
                "min_steps": online.reasoner.min_steps,
                "max_steps": online.reasoner.max_steps,
                "reasoning_reward": REASONING_REWARD,
                "gamma_think": GAMMA_THINK,
                "reasoning_loss": online.reasoning_loss,
                "residual_connections": online.reasoner.residual_connections,
                "wrap_latent": online.reasoner.wrap_latent,
                "initial_state": "zero_feedback_with_learned_input_offset",
                "input_norm": "RMSNorm",
                "input_projection": "gated_mlp_after_rms_norm",
                "input_mlp_output_scale": online.reasoner.input_mlp_output_scale,
                "norm_eps": online.reasoner.norm.eps,
                "explicit_time_embeddings": False,
                "num_actions": MAX_ACTIONS,
                "hidden_dim": backbone.hidden_dim,
            },
            indent=2,
        )
        + "\n"
    )
    api = HfApi()
    repository = api.create_repo(
        repo_id=MODEL_ID, repo_type="model", private=False, exist_ok=True
    )
    api.upload_folder(
        repo_id=repository.repo_id,
        repo_type="model",
        folder_path=directory,
        commit_message="Upload reasoning model, tokenizers and inference settings",
    )
print(f"Uploaded checkpoint bundle to {repository}")